In [ ]:
import pandas as pd

# ==============================
# Fill in your CSV information
# ==============================

FILE_PATH = "/Users/zhouyuxuan/Downloads/ot_delaycause1_DL (1)/Airline_Delay_Cause.csv"         
MONTH_COLUMN = "month"      
AIRPORT_COLUMN = "airport_name"    


# ==============================
# Read data
# ==============================

df = pd.read_csv(FILE_PATH)


# ==============================
# 1. Count airports for each month
# ==============================

monthly_count = (
    df.groupby(MONTH_COLUMN)[AIRPORT_COLUMN]
      .nunique()
      .reset_index(name="airport_count")
)

print("\n===== Airport count by month =====")
print(monthly_count.to_string(index=False))


# ==============================
# 2. Check whether all months
#    contain the same airports
# ==============================

airport_sets = (
    df.groupby(MONTH_COLUMN)[AIRPORT_COLUMN]
      .apply(lambda x: set(x.dropna()))
)

months = airport_sets.index
first_month = months[0]
first_set = airport_sets.iloc[0]

print("\n===== Airport consistency =====")

all_same = True

for month, airports in airport_sets.items():

    if airports == first_set:
        print(f"{month}: SAME")
    else:
        all_same = False

        print(f"{month}: DIFFERENT")

        missing = first_set - airports
        extra = airports - first_set

        print("  Missing airports:", sorted(missing))
        print("  Extra airports:", sorted(extra))


# ==============================
# 3. Overall result
# ==============================

print("\n===== Overall result =====")

if all_same:
    print("All months contain exactly the same airports.")
else:
    print("The airports are NOT exactly the same across all months.")


===== Airport count by month =====
 month  airport_count
     1            359
     2            362
     3            361
     4            361
     5            364
     6            368
     7            360
     8            363
     9            360
    10            361
    11            357
    12            359

===== Airport consistency =====
1: SAME
2: DIFFERENT
  Missing airports: []
  Extra airports: ['Lewisburg, WV: Greenbrier Valley', 'Paducah, KY: Barkley Regional', 'Staunton, VA: Shenandoah Valley Regional']
3: DIFFERENT
  Missing airports: ['Binghamton, NY: Greater Binghamton/Edwin A. Link Field']
  Extra airports: ['Lewisburg, WV: Greenbrier Valley', 'Paducah, KY: Barkley Regional', 'Staunton, VA: Shenandoah Valley Regional']
4: DIFFERENT
  Missing airports: ['Binghamton, NY: Greater Binghamton/Edwin A. Link Field']
  Extra airports: ['Lewisburg, WV: Greenbrier Valley', 'Paducah, KY: Barkley Regional', 'Staunton, VA: Shenandoah Valley Regional']
5: DIFFERENT
  Missin

In [3]:
# Extract June airports
june_airports = (
    df[df[MONTH_COLUMN] == 6][AIRPORT_COLUMN]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

# Rename column
june_airports = june_airports.to_frame(name="airport_name")

# Save to Downloads
import os

output_path = os.path.expanduser("~/Downloads/june_airports.xlsx")
june_airports.to_excel(output_path, index=False)

print(f"June has {len(june_airports)} unique airports.")
print(f"Saved to: {output_path}")

June has 368 unique airports.
Saved to: /Users/zhouyuxuan/Downloads/june_airports.xlsx


In [10]:
pip install pandas requests beautifulsoup4 rapidfuzz openpyxl tqdm

Note: you may need to restart the kernel to use updated packages.


In [16]:
# ============================================================
# NOAA GSOM downloader - corrected version
# NO API TOKEN REQUIRED
# ============================================================

import pandas as pd
import requests
from io import StringIO
from pathlib import Path
from tqdm import tqdm
import time
import re
from rapidfuzz import process, fuzz


# ============================================================
# 1. Paths
# ============================================================



AIRPORT_FILE = "/Users/zhouyuxuan/Downloads/june_airports.xlsx"

OUTPUT_DIR = Path("/Users/zhouyuxuan/Downloads")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MATCH_FILE = OUTPUT_DIR / "airport_station_matches.csv"
OUTPUT_FILE = OUTPUT_DIR / "airport_gsom_2020_2026.csv"


# ============================================================
# 2. NOAA GSOM
# ============================================================

GSOM_BASE = (
    "https://www.ncei.noaa.gov/"
    "data/global-summary-of-the-month/access/"
)


# ============================================================
# 3. Read airport file
# ============================================================

airports = pd.read_excel(AIRPORT_FILE)

print("Airports:", len(airports))


# ============================================================
# 4. Parse airport name
# ============================================================

def parse_airport_name(x):

    x = str(x).strip()

    m = re.match(
        r"^(.*?),\s*([A-Z]{2}):\s*(.*)$",
        x
    )

    if m:

        return (
            m.group(1).strip(),
            m.group(2).strip(),
            m.group(3).strip()
        )

    return "", "", x


airports[
    ["city", "state", "airport"]
] = airports["airport_name"].apply(
    lambda x: pd.Series(
        parse_airport_name(x)
    )
)


# ============================================================
# 5. Download GHCN station metadata
# ============================================================

STATION_URL = (
    "https://www.ncei.noaa.gov/"
    "pub/data/ghcn/daily/ghcnd-stations.txt"
)

print("\nDownloading station metadata...")

r = requests.get(
    STATION_URL,
    timeout=120
)

r.raise_for_status()

station_text = r.text


# ============================================================
# 6. Parse stations
# ============================================================

stations = []

for line in station_text.splitlines():

    if len(line) < 70:
        continue

    station_id = line[0:11].strip()
    latitude = line[12:20].strip()
    longitude = line[21:30].strip()
    elevation = line[31:37].strip()
    state = line[38:40].strip()
    name = line[41:71].strip()

    if station_id.startswith("US"):

        stations.append({
            "station_id": station_id,
            "latitude": latitude,
            "longitude": longitude,
            "elevation": elevation,
            "state": state,
            "station_name": name
        })


stations = pd.DataFrame(stations)


# ============================================================
# 7. Normalize
# ============================================================

def normalize(x):

    x = str(x).upper()

    x = x.replace("&", "AND")

    x = re.sub(
        r"[^A-Z0-9 ]",
        " ",
        x
    )

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x.strip()


stations["name_norm"] = (
    stations["station_name"]
    .apply(normalize)
)

stations["state_norm"] = (
    stations["state"]
    .astype(str)
    .str.upper()
)


# ============================================================
# 8. Test whether a station actually has GSOM
# ============================================================

session = requests.Session()

def download_gsom(station_id):

    url = GSOM_BASE + station_id + ".csv"

    try:

        response = session.get(
            url,
            timeout=30
        )

        if response.status_code != 200:
            return None

        if len(response.content) < 100:
            return None

        text = response.text

        if "STATION" not in text[:500]:
            return None

        df = pd.read_csv(
            StringIO(text)
        )

        if "DATE" not in df.columns:
            return None

        return df

    except Exception:

        return None


# ============================================================
# 9. Find GSOM station
# ============================================================

def find_gsom_station(row):

    city = normalize(row["city"])
    airport = normalize(row["airport"])
    state = row["state"].upper()

    candidates = stations[
        stations["state_norm"] == state
    ].copy()

    if len(candidates) == 0:
        return None, None, 0

    # ----------------------------------------
    # Try airport name first
    # ----------------------------------------

    queries = [
        f"{city} {airport}",
        airport,
        city
    ]

    checked = set()

    for query in queries:

        matches = process.extract(
            query,
            candidates["name_norm"].tolist(),
            scorer=fuzz.token_set_ratio,
            limit=20
        )

        for match_name, score, idx in matches:

            station = candidates.iloc[idx]

            station_id = station["station_id"]

            if station_id in checked:
                continue

            checked.add(station_id)

            # ------------------------------------------------
            # IMPORTANT:
            # Test whether this station actually exists in GSOM
            # ------------------------------------------------

            test_df = download_gsom(
                station_id
            )

            if test_df is not None:

                return (
                    station_id,
                    station["station_name"],
                    score
                )

    return None, None, 0


# ============================================================
# 10. Match airport -> REAL GSOM station
# ============================================================

match_results = []

print("\nFinding stations that actually have GSOM data...\n")

for _, row in tqdm(
    airports.iterrows(),
    total=len(airports),
    desc="Matching"
):

    station_id, station_name, score = (
        find_gsom_station(row)
    )

    match_results.append({

        "noaa_station_id":
            station_id,

        "noaa_station_name":
            station_name,

        "match_score":
            score
    })


match_df = pd.DataFrame(
    match_results
)


airports = pd.concat(
    [
        airports.reset_index(drop=True),
        match_df
    ],
    axis=1
)


# ============================================================
# 11. Save matching table
# ============================================================

airports.to_csv(
    MATCH_FILE,
    index=False
)

print(
    "\nSaved:",
    MATCH_FILE
)


# ============================================================
# 12. Download final GSOM data
# ============================================================

all_data = []

failed = []


for _, row in tqdm(
    airports.iterrows(),
    total=len(airports),
    desc="Downloading GSOM"
):

    station_id = row["noaa_station_id"]

    if pd.isna(station_id):

        failed.append(
            row["airport_name"]
        )

        continue

    station_id = str(
        station_id
    )

    df = download_gsom(
        station_id
    )

    if df is None:

        failed.append(
            row["airport_name"]
        )

        continue


    # ========================================================
    # Date
    # ========================================================

    df["DATE"] = pd.to_datetime(
        df["DATE"].astype(str),
        format="%Y%m",
        errors="coerce"
    )


    # ========================================================
    # 2020-2026
    # ========================================================

    df = df[
        (df["DATE"] >= "2020-01-01") &
        (df["DATE"] <= "2026-12-31")
    ].copy()


    if len(df) == 0:
        continue


    # ========================================================
    # Airport information
    # ========================================================

    df["airport_name"] = (
        row["airport_name"]
    )

    df["city"] = (
        row["city"]
    )

    df["state"] = (
        row["state"]
    )

    df["noaa_station_id"] = (
        station_id
    )

    df["noaa_station_name"] = (
        row["noaa_station_name"]
    )


    all_data.append(df)

    time.sleep(0.1)


# ============================================================
# 13. Check
# ============================================================

print("\n===================================")
print("DOWNLOAD SUMMARY")
print("===================================")

print(
    "Airports:",
    len(airports)
)

print(
    "Successful:",
    len(all_data)
)

print(
    "Failed:",
    len(failed)
)


# ============================================================
# 14. Combine
# ============================================================

if len(all_data) == 0:

    raise RuntimeError(
        "Still no GSOM stations found."
    )


result = pd.concat(
    all_data,
    ignore_index=True
)


# ============================================================
# 15. Reorder columns
# ============================================================

first_columns = [

    "airport_name",
    "city",
    "state",

    "noaa_station_id",
    "noaa_station_name",

    "DATE"
]


other_columns = [

    c for c in result.columns
    if c not in first_columns
]


result = result[
    first_columns + other_columns
]


# ============================================================
# 16. Save
# ============================================================

result.to_csv(
    OUTPUT_FILE,
    index=False
)


# ============================================================
# 17. Failed airports
# ============================================================

FAILED_FILE = (
    OUTPUT_DIR /
    "failed_airports.csv"
)


if failed:

    pd.DataFrame({
        "airport_name": failed
    }).to_csv(
        FAILED_FILE,
        index=False
    )


# ============================================================
# 18. Final information
# ============================================================

print("\n===================================")
print("DONE")
print("===================================")

print(
    "Output:",
    OUTPUT_FILE
)

print(
    "Rows:",
    len(result)
)

print(
    "Airports with data:",
    result["airport_name"].nunique()
)

print(
    "Date:",
    result["DATE"].min(),
    "->",
    result["DATE"].max()
)

print(
    "\nFiles saved in:",
    OUTPUT_DIR
)

Airports: 368


Finding stations that actually have GSOM data...



Matching: 100%|██████████| 368/368 [03:39<00:00,  1.67it/s]



Saved: /Users/zhouyuxuan/Downloads/airport_station_matches.csv



DOWNLOAD SUMMARY
Airports: 368
Successful: 0
Failed: 13


RuntimeError: Still no GSOM stations found.

In [20]:
import pandas as pd
import re
from rapidfuzz import process, fuzz

AIRPORT_FILE = "/Users/zhouyuxuan/Downloads/june_airports.xlsx"

airports = pd.read_excel(AIRPORT_FILE)

# ============================================================
# GSOM station IDs that REALLY exist
# ============================================================

gsom_set = set(gsom_files)

print("Real GSOM stations:", len(gsom_set))


# ============================================================
# Parse airport name
# ============================================================

def parse_airport_name(x):

    x = str(x).strip()

    m = re.match(
        r"^(.*?),\s*([A-Z]{2}):\s*(.*)$",
        x
    )

    if m:
        return (
            m.group(1).strip(),
            m.group(2).strip(),
            m.group(3).strip()
        )

    return "", "", x


airports[
    ["city", "state", "airport"]
] = airports["airport_name"].apply(
    lambda x: pd.Series(parse_airport_name(x))
)


# ============================================================
# We need station names
# Download GHCN metadata ONLY for names/locations.
# But keep only stations that actually exist in GSOM.
# ============================================================

STATION_URL = (
    "https://www.ncei.noaa.gov/"
    "pub/data/ghcn/daily/ghcnd-stations.txt"
)

import requests

response = requests.get(
    STATION_URL,
    timeout=120
)

response.raise_for_status()

station_text = response.text


stations = []

for line in station_text.splitlines():

    if len(line) < 70:
        continue

    station_id = line[0:11].strip()

    # ONLY stations that have GSOM
    if station_id not in gsom_set:
        continue

    latitude = line[12:20].strip()
    longitude = line[21:30].strip()
    elevation = line[31:37].strip()
    state = line[38:40].strip()
    name = line[41:71].strip()

    if station_id.startswith("US"):

        stations.append({
            "station_id": station_id,
            "latitude": latitude,
            "longitude": longitude,
            "elevation": elevation,
            "state": state,
            "station_name": name
        })


stations = pd.DataFrame(stations)

print(
    "US GSOM stations with metadata:",
    len(stations)
)


# ============================================================
# Normalize text
# ============================================================

def normalize(x):

    x = str(x).upper()

    x = x.replace("&", " AND ")

    x = re.sub(
        r"[^A-Z0-9 ]",
        " ",
        x
    )

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x.strip()


stations["name_norm"] = (
    stations["station_name"]
    .apply(normalize)
)

stations["state_norm"] = (
    stations["state"]
    .astype(str)
    .str.upper()
)


# ============================================================
# Airport keywords
# ============================================================

airport_keywords = [
    "AIRPORT",
    "AP",
    "INTL",
    "INTERNATIONAL",
    "REGIONAL",
    "FIELD",
    "MUNICIPAL"
]


# ============================================================
# Match
# ============================================================

def match_airport(row):

    city = normalize(row["city"])
    airport = normalize(row["airport"])
    state = row["state"].upper()

    candidates = stations[
        stations["state_norm"] == state
    ].copy()

    if len(candidates) == 0:
        return None

    # --------------------------------------------------------
    # Score every candidate
    # --------------------------------------------------------

    scores = []

    query = f"{city} {airport}"

    for idx, station in candidates.iterrows():

        station_name = station["name_norm"]

        # Basic fuzzy similarity
        score = fuzz.token_set_ratio(
            query,
            station_name
        )

        # City appears in station name
        if city and city in station_name:
            score += 15

        # Airport-related station
        if any(
            word in station_name.split()
            for word in airport_keywords
        ):
            score += 10

        # Exact city + airport word
        if city in station_name:
            score += 10

        scores.append(
            (idx, score)
        )

    # Highest score
    scores.sort(
        key=lambda x: x[1],
        reverse=True
    )

    best_idx, best_score = scores[0]

    best = stations.loc[best_idx]

    return {
        "noaa_station_id":
            best["station_id"],

        "noaa_station_name":
            best["station_name"],

        "latitude":
            best["latitude"],

        "longitude":
            best["longitude"],

        "elevation":
            best["elevation"],

        "match_score":
            best_score
    }


# ============================================================
# Run matching
# ============================================================

results = []

for _, row in airports.iterrows():

    result = match_airport(row)

    if result is None:

        results.append({
            "noaa_station_id": None,
            "noaa_station_name": None,
            "latitude": None,
            "longitude": None,
            "elevation": None,
            "match_score": 0
        })

    else:

        results.append(result)


match_df = pd.DataFrame(results)

final_matches = pd.concat(
    [
        airports.reset_index(drop=True),
        match_df
    ],
    axis=1
)


# ============================================================
# Save
# ============================================================

MATCH_FILE = (
    "/Users/zhouyuxuan/Downloads/"
    "airport_station_matches.csv"
)

final_matches.to_csv(
    MATCH_FILE,
    index=False
)

print("\nSaved:")
print(MATCH_FILE)

print(
    "\nMatched airports:",
    final_matches["noaa_station_id"].notna().sum()
)

print(
    "Unmatched:",
    final_matches["noaa_station_id"].isna().sum()
)

Real GSOM stations: 125008
US GSOM stations with metadata: 72652

Saved:
/Users/zhouyuxuan/Downloads/airport_station_matches.csv

Matched airports: 360
Unmatched: 8


In [21]:
final_matches[
    [
        "airport_name",
        "noaa_station_id",
        "noaa_station_name",
        "match_score"
    ]
].head(30)

,airport_name,noaa_station_id,noaa_station_name,match_score
0,"Aberdeen, SD: Aberdeen Regional",USW00014929,ABERDEEN,125.000000
1,"Abilene, TX: Abilene Regional",USW00013962,ABILENE RGNL AP,112.419355
2,"Adak Island, AK: Adak",USW00025704,ADAK AP,82.727273
3,"Aguadilla, PR: Rafael Hernandez",None,None,0.000000
4,"Akron, OH: Akron-Canton Regional",USC00330061,AKRON,125.000000
5,"Albany, GA: Southwest Georgia Regional",USW00013869,ALBANY SW GEORGIA RGNL AP,110.862069
6,"Albany, NY: Albany International",USC00300047,ALBANY,125.000000
7,"Albuquerque, NM: Albuquerque International Sun...",US1NMBR0100,ALBUQUERQUE 6.6 E,109.615385
8,"Alexandria, LA: Alexandria International",USC00160098,ALEXANDRIA,125.000000
9,"Allentown/Bethlehem/Easton, PA: Lehigh Valley ...",USC00360629,BETHLEHEM,100.000000


In [22]:
import pandas as pd
import numpy as np
import requests
import re
import os
from pathlib import Path
from difflib import SequenceMatcher

# ============================================================
# 0. PATHS
# ============================================================

AIRPORT_FILE = Path("/Users/zhouyuxuan/Downloads/june_airports.xlsx")

OUTPUT_DIR = Path("/Users/zhouyuxuan/Downloads/airport_weather")
OUTPUT_DIR.mkdir(exist_ok=True)

MATCH_FILE = OUTPUT_DIR / "airport_station_matches.csv"
FINAL_FILE = OUTPUT_DIR / "airport_GSOM_2020_2026.csv"

# NOAA GSOM
GSOM_URL = (
    "https://www.ncei.noaa.gov/data/"
    "global-summary-of-the-month/access/"
)

GHCN_STATIONS_URL = (
    "https://www.ncei.noaa.gov/pub/data/ghcn/daily/"
    "ghcnd-stations.txt"
)


# ============================================================
# 1. READ AIRPORT LIST
# ============================================================

airports = pd.read_excel(AIRPORT_FILE)

print("Number of airports:", len(airports))
print(airports.head())


# ============================================================
# 2. PARSE airport_name
# Example:
# "Moline, IL: Quad Cities International"
# ============================================================

def parse_airport_name(x):

    x = str(x).strip()

    m = re.match(
        r"^(.*?),\s*([A-Z]{2}):\s*(.*)$",
        x
    )

    if m:
        return pd.Series({
            "city": m.group(1).strip(),
            "state": m.group(2).strip(),
            "airport": m.group(3).strip()
        })

    return pd.Series({
        "city": x,
        "state": "",
        "airport": ""
    })


airports[["city", "state", "airport"]] = (
    airports["airport_name"]
    .apply(parse_airport_name)
)


# ============================================================
# 3. GET ALL REAL GSOM STATION FILES
#
# IMPORTANT:
# We only use stations that actually have a GSOM CSV.
# ============================================================

print("\nDownloading NOAA GSOM directory...")

r = requests.get(GSOM_URL, timeout=60)
r.raise_for_status()

html = r.text

gsom_files = re.findall(
    r'href="([A-Z0-9]+)\.csv"',
    html,
    flags=re.IGNORECASE
)

gsom_files = sorted(set(gsom_files))

gsom_set = set(gsom_files)

print("GSOM station files:", len(gsom_files))
print("Example:", gsom_files[:10])


# ============================================================
# 4. DOWNLOAD GHCN STATION METADATA
#
# This is ONLY used to get:
# station ID
# station name
# latitude
# longitude
# elevation
#
# We DO NOT assume every GHCN station has GSOM.
# ============================================================

print("\nDownloading GHCN station metadata...")

r = requests.get(GHCN_STATIONS_URL, timeout=120)
r.raise_for_status()

lines = r.text.splitlines()

stations = []

for line in lines:

    if len(line) < 85:
        continue

    station_id = line[0:11].strip()

    if station_id not in gsom_set:
        continue

    try:
        lat = float(line[12:20])
        lon = float(line[21:30])
        elev = float(line[31:37])
    except:
        lat = np.nan
        lon = np.nan
        elev = np.nan

    station_name = line[41:71].strip()

    stations.append({
        "station_id": station_id,
        "station_name": station_name,
        "latitude": lat,
        "longitude": lon,
        "elevation": elev
    })


stations = pd.DataFrame(stations)

print("GSOM stations with metadata:", len(stations))
print(stations.head())


# ============================================================
# 5. NORMALIZE TEXT
# ============================================================

def normalize_text(x):

    x = str(x).upper()

    # common abbreviations
    replacements = {
        "INTERNATIONAL": "INTL",
        "AIRPORT": "AP",
        "REGIONAL": "REG",
        "MUNICIPAL": "MUNI",
        "COUNTY": "",
        "FIELD": "FLD",
        "SAWYER": "",
        "STATION": "",
    }

    for a, b in replacements.items():
        x = x.replace(a, b)

    # punctuation -> space
    x = re.sub(r"[^A-Z0-9 ]", " ", x)

    # collapse spaces
    x = re.sub(r"\s+", " ", x).strip()

    return x


# ============================================================
# 6. REMOVE BAD "DISTANCE FROM CITY" STATIONS
#
# Examples:
# BELLEVILLE 2.2 SE
# CHICAGO 6.6 E
# etc.
# ============================================================

def is_distance_station(name):

    name = str(name).upper()

    patterns = [
        r"\d+(\.\d+)?\s*[NSEW]$",
        r"\d+(\.\d+)?\s*[NSEW]\s*$",
        r"\d+(\.\d+)?\s*(N|S|E|W)\s*$",
    ]

    for p in patterns:
        if re.search(p, name):
            return True

    return False


stations["distance_station"] = (
    stations["station_name"]
    .apply(is_distance_station)
)


# ============================================================
# 7. STATION TYPE
# ============================================================

AIRPORT_WORDS = [
    "AIRPORT",
    " AP",
    "INTL",
    "INTERNATIONAL",
    "REGIONAL",
    "FIELD",
    "MUNICIPAL",
    "NAS",
    "AFB",
    "AIR FORCE",
]


def station_type(name):

    name_upper = str(name).upper()

    if is_distance_station(name_upper):
        return "DISTANCE_FROM_CITY"

    for word in AIRPORT_WORDS:
        if word in name_upper:
            return "AIRPORT"

    return "CITY_OR_OTHER"


stations["station_type"] = (
    stations["station_name"]
    .apply(station_type)
)


# ============================================================
# 8. TEXT SIMILARITY
# ============================================================

def similarity(a, b):

    a = normalize_text(a)
    b = normalize_text(b)

    return SequenceMatcher(
        None,
        a,
        b
    ).ratio()


# ============================================================
# 9. MATCH ONE AIRPORT
# ============================================================

def match_airport(row):

    city = str(row["city"])
    airport = str(row["airport"])
    state = str(row["state"])

    # --------------------------------------------------------
    # Only same-state stations
    # --------------------------------------------------------

    candidates = stations.copy()

    # US station IDs start with US
    candidates = candidates[
        candidates["station_id"].str.startswith("US")
    ]

    # --------------------------------------------------------
    # Remove distance stations
    # --------------------------------------------------------

    candidates = candidates[
        ~candidates["distance_station"]
    ]

    # --------------------------------------------------------
    # Airport station name
    # --------------------------------------------------------

    airport_candidates = candidates[
        candidates["station_type"] == "AIRPORT"
    ]

    # We don't know station state from this file directly,
    # so first score all airport/city names.
    # --------------------------------------------------------

    results = []

    for _, s in candidates.iterrows():

        station_name = s["station_name"]

        city_score = similarity(
            city,
            station_name
        )

        airport_score = similarity(
            airport,
            station_name
        )

        combined_score = (
            0.60 * city_score +
            0.40 * airport_score
        )

        name_upper = station_name.upper()

        # Strong bonus for airport stations
        airport_bonus = 0

        if s["station_type"] == "AIRPORT":
            airport_bonus += 0.20

        # Stronger if city name appears
        city_norm = normalize_text(city)
        station_norm = normalize_text(station_name)

        if city_norm and city_norm in station_norm:
            combined_score += 0.20

        # If airport name has a distinctive word
        airport_words = [
            w for w in normalize_text(airport).split()
            if len(w) >= 4
        ]

        for word in airport_words:

            if word in station_norm:
                combined_score += 0.08

        combined_score += airport_bonus

        results.append({
            "station_id": s["station_id"],
            "station_name": station_name,
            "station_type": s["station_type"],
            "latitude": s["latitude"],
            "longitude": s["longitude"],
            "elevation": s["elevation"],
            "match_score": combined_score
        })

    results = pd.DataFrame(results)

    if len(results) == 0:
        return None

    results = results.sort_values(
        "match_score",
        ascending=False
    )

    return results.iloc[0]


# ============================================================
# 10. MATCH ALL AIRPORTS
# ============================================================

print("\nMatching airports to GSOM stations...")

matches = []

for i, row in airports.iterrows():

    best = match_airport(row)

    if best is None:

        matches.append({
            "airport_name": row["airport_name"],
            "city": row["city"],
            "state": row["state"],
            "airport": row["airport"],
            "station_id": None,
            "station_name": None,
            "station_type": None,
            "latitude": np.nan,
            "longitude": np.nan,
            "elevation": np.nan,
            "match_score": np.nan
        })

    else:

        matches.append({
            "airport_name": row["airport_name"],
            "city": row["city"],
            "state": row["state"],
            "airport": row["airport"],
            **best.to_dict()
        })

matches = pd.DataFrame(matches)


# ============================================================
# 11. SAVE MATCH RESULTS
# ============================================================

matches.to_csv(
    MATCH_FILE,
    index=False
)

print("\nMatching finished.")

print(
    matches[
        [
            "airport_name",
            "station_id",
            "station_name",
            "station_type",
            "match_score"
        ]
    ].head(20)
)

print(
    "\nSaved:",
    MATCH_FILE
)


# ============================================================
# 12. SHOW POTENTIALLY BAD MATCHES
# ============================================================

print("\nPotentially questionable matches:")

bad_matches = matches[
    (matches["station_id"].isna()) |
    (matches["match_score"] < 0.65) |
    (matches["station_type"] != "AIRPORT")
]

print(
    bad_matches[
        [
            "airport_name",
            "station_id",
            "station_name",
            "station_type",
            "match_score"
        ]
    ].to_string(index=False)
)


# ============================================================
# 13. DOWNLOAD GSOM DATA
#
# Each station has:
#
# https://www.ncei.noaa.gov/data/
# global-summary-of-the-month/access/
# STATION_ID.csv
#
# ============================================================

def download_gsom(station_id):

    url = f"{GSOM_URL}{station_id}.csv"

    output_file = OUTPUT_DIR / f"{station_id}.csv"

    # already downloaded
    if output_file.exists():

        print(
            f"Already exists: {station_id}"
        )

        return output_file

    try:

        response = requests.get(
            url,
            timeout=60
        )

        if response.status_code != 200:

            print(
                f"FAILED {station_id}: "
                f"HTTP {response.status_code}"
            )

            return None

        output_file.write_bytes(
            response.content
        )

        print(
            f"Downloaded: {station_id}"
        )

        return output_file

    except Exception as e:

        print(
            f"ERROR {station_id}: {e}"
        )

        return None


# ============================================================
# 14. DOWNLOAD ALL UNIQUE STATIONS
# ============================================================

station_ids = (
    matches["station_id"]
    .dropna()
    .unique()
    .tolist()
)

print(
    "\nStations to download:",
    len(station_ids)
)

downloaded_files = []

for station_id in station_ids:

    f = download_gsom(station_id)

    if f is not None:
        downloaded_files.append(f)


print(
    "\nDownloaded files:",
    len(downloaded_files)
)


# ============================================================
# 15. READ + FILTER 2020–2026
# ============================================================

all_weather = []

for file in downloaded_files:

    try:

        df = pd.read_csv(file)

        if "DATE" not in df.columns:
            print(
                "No DATE:",
                file.name
            )
            continue

        # DATE is usually YYYY-MM
        df["DATE"] = pd.to_datetime(
            df["DATE"],
            errors="coerce"
        )

        # Keep 2020-01 through 2026-12
        df = df[
            (df["DATE"] >= "2020-01-01") &
            (df["DATE"] <= "2026-12-31")
        ].copy()

        if len(df) > 0:
            all_weather.append(df)

    except Exception as e:

        print(
            f"ERROR reading {file.name}: {e}"
        )


# ============================================================
# 16. COMBINE WEATHER DATA
# ============================================================

if len(all_weather) == 0:

    raise RuntimeError(
        "No weather data found."
    )

weather = pd.concat(
    all_weather,
    ignore_index=True
)

print(
    "\nRaw weather rows:",
    len(weather)
)


# ============================================================
# 17. ADD AIRPORT INFORMATION
# ============================================================

matches_for_merge = matches[
    [
        "airport_name",
        "city",
        "state",
        "airport",
        "station_id",
        "station_name",
        "station_type",
        "latitude",
        "longitude",
        "elevation",
        "match_score"
    ]
].copy()

weather = weather.merge(
    matches_for_merge,
    on="station_id",
    how="left"
)


# ============================================================
# 18. REORDER COLUMNS
# ============================================================

first_columns = [
    "airport_name",
    "city",
    "state",
    "airport",
    "station_id",
    "station_name",
    "station_type",
    "latitude",
    "longitude",
    "elevation",
    "match_score",
    "DATE"
]

remaining_columns = [
    c for c in weather.columns
    if c not in first_columns
]

weather = weather[
    first_columns + remaining_columns
]


# ============================================================
# 19. SAVE FINAL CSV
# ============================================================

weather.to_csv(
    FINAL_FILE,
    index=False
)

print("\n====================================")
print("DONE")
print("====================================")

print(
    "Final file:",
    FINAL_FILE
)

print(
    "Rows:",
    len(weather)
)

print(
    "Airports:",
    weather["airport_name"].nunique()
)

print(
    "Stations:",
    weather["station_id"].nunique()
)

print(
    "Date range:",
    weather["DATE"].min(),
    "to",
    weather["DATE"].max()
)

print("\nColumns:")
print(weather.columns.tolist())

Number of airports: 368
                       airport_name
0   Aberdeen, SD: Aberdeen Regional
1     Abilene, TX: Abilene Regional
2             Adak Island, AK: Adak
3   Aguadilla, PR: Rafael Hernandez
4  Akron, OH: Akron-Canton Regional

GSOM station files: 126536
Example: ['ACW00011604', 'ACW00011647', 'AE000041196', 'AEM00041194', 'AEM00041217', 'AEM00041218', 'AF000040930', 'AFM00040938', 'AFM00040948', 'AFM00040990']

GSOM stations with metadata: 126536
    station_id           station_name  latitude  longitude  elevation
0  ACW00011604  ST JOHNS COOLIDGE FLD   17.1167   -61.7833       10.1
1  ACW00011647               ST JOHNS   17.1333   -61.7833       19.2
2  AE000041196    SHARJAH INTER. AIRP   25.3330    55.5170       34.0
3  AEM00041194             DUBAI INTL   25.2550    55.3640       10.4
4  AEM00041217         ABU DHABI INTL   24.4330    54.6510       26.8

Matching airports to GSOM stations...

Matching finished.
                                         airport_name   

KeyError: 'station_id'

In [24]:
from pathlib import Path
import pandas as pd

# ============================================================
# PATH
# ============================================================

DATA_DIR = Path("/Users/zhouyuxuan/Downloads/airport_weather")

START_DATE = pd.Timestamp("2020-01-01")
END_DATE = pd.Timestamp("2026-09-19")

# 输出
FILTERED_DIR = DATA_DIR / "filtered_2020_to_now"
FILTERED_DIR.mkdir(exist_ok=True)

REPORT_FILE = DATA_DIR / "date_coverage_report.csv"


# ============================================================
# FIND ALL DOWNLOADED GSOM FILES
# ============================================================

csv_files = [
    f for f in DATA_DIR.glob("*.csv")
    if f.name != "airport_station_matches.csv"
    and f.name != "airport_GSOM_2020_2026.csv"
]

print("Found CSV files:", len(csv_files))


# ============================================================
# CHECK EACH STATION
# ============================================================

report = []
filtered_data = []

for file in csv_files:

    station_id = file.stem

    try:
        df = pd.read_csv(file)

        # ----------------------------
        # Check DATE column
        # ----------------------------
        if "DATE" not in df.columns:

            report.append({
                "station_id": station_id,
                "file": file.name,
                "status": "NO_DATE_COLUMN",
                "original_start": None,
                "original_end": None,
                "filtered_rows": 0,
                "missing_before_2020": None,
                "missing_after_2026_09_19": None
            })

            continue

        # ----------------------------
        # Convert DATE
        # ----------------------------
        df["DATE"] = pd.to_datetime(
            df["DATE"],
            errors="coerce"
        )

        df = df.dropna(subset=["DATE"])

        if len(df) == 0:

            report.append({
                "station_id": station_id,
                "file": file.name,
                "status": "NO_VALID_DATE",
                "original_start": None,
                "original_end": None,
                "filtered_rows": 0,
                "missing_before_2020": None,
                "missing_after_2026_09_19": None
            })

            continue

        # ----------------------------
        # Original date range
        # ----------------------------
        original_start = df["DATE"].min()
        original_end = df["DATE"].max()

        # ----------------------------
        # Filter
        # ----------------------------
        filtered = df[
            (df["DATE"] >= START_DATE) &
            (df["DATE"] <= END_DATE)
        ].copy()

        # Make sure station_id exists
        filtered["station_id"] = station_id

        # ----------------------------
        # Determine coverage
        # ----------------------------

        if original_start > START_DATE:
            before_status = "STARTS_AFTER_2020"

        else:
            before_status = "OK"

        if original_end < END_DATE:
            after_status = "ENDS_BEFORE_2026_09_19"

        else:
            after_status = "OK"

        # ----------------------------
        # Overall status
        # ----------------------------

        if (
            before_status == "OK"
            and after_status == "OK"
        ):
            status = "FULL_COVERAGE"

        elif (
            before_status != "OK"
            and after_status != "OK"
        ):
            status = "MISSING_START_AND_END"

        elif before_status != "OK":
            status = "MISSING_START"

        else:
            status = "MISSING_END"

        # ----------------------------
        # Save filtered data
        # ----------------------------

        if len(filtered) > 0:

            output_file = (
                FILTERED_DIR /
                f"{station_id}_2020_to_2026-09-19.csv"
            )

            filtered.to_csv(
                output_file,
                index=False
            )

            filtered_data.append(filtered)

        # ----------------------------
        # Report
        # ----------------------------

        report.append({
            "station_id": station_id,
            "file": file.name,
            "status": status,
            "original_start": original_start.date(),
            "original_end": original_end.date(),
            "filtered_rows": len(filtered),
            "missing_before_2020": (
                original_start > START_DATE
            ),
            "missing_after_2026_09_19": (
                original_end < END_DATE
            )
        })

    except Exception as e:

        report.append({
            "station_id": station_id,
            "file": file.name,
            "status": f"ERROR: {e}",
            "original_start": None,
            "original_end": None,
            "filtered_rows": 0,
            "missing_before_2020": None,
            "missing_after_2026_09_19": None
        })


# ============================================================
# SAVE REPORT
# ============================================================

report_df = pd.DataFrame(report)

report_df.to_csv(
    REPORT_FILE,
    index=False
)


# ============================================================
# COMBINE ALL FILTERED DATA
# ============================================================

if filtered_data:

    final_data = pd.concat(
        filtered_data,
        ignore_index=True
    )

    FINAL_FILTERED_FILE = (
        DATA_DIR /
        "airport_GSOM_2020_to_2026-09-19.csv"
    )

    final_data.to_csv(
        FINAL_FILTERED_FILE,
        index=False
    )

else:

    FINAL_FILTERED_FILE = None


# ============================================================
# PRINT SUMMARY
# ============================================================

print("\n======================================")
print("DATE FILTER FINISHED")
print("======================================")

print(
    "Required period:",
    START_DATE.date(),
    "to",
    END_DATE.date()
)

print(
    "Stations processed:",
    len(report_df)
)

print(
    "Full coverage:",
    (report_df["status"] == "FULL_COVERAGE").sum()
)

print(
    "Missing start:",
    report_df["status"].isin([
        "MISSING_START",
        "MISSING_START_AND_END"
    ]).sum()
)

print(
    "Missing end:",
    report_df["status"].isin([
        "MISSING_END",
        "MISSING_START_AND_END"
    ]).sum()
)

print(
    "No valid date:",
    report_df["status"].isin([
        "NO_DATE_COLUMN",
        "NO_VALID_DATE"
    ]).sum()
)


# ============================================================
# SHOW PROBLEMATIC STATIONS
# ============================================================

print("\n======================================")
print("STATIONS THAT DO NOT COVER 2020-09-19")
print("======================================")

problem = report_df[
    report_df["status"] != "FULL_COVERAGE"
]

print(
    problem[
        [
            "station_id",
            "status",
            "original_start",
            "original_end",
            "filtered_rows"
        ]
    ].to_string(index=False)
)


print("\n======================================")
print("FILES")
print("======================================")

print(
    "Filtered individual files:",
    FILTERED_DIR
)

print(
    "Date coverage report:",
    REPORT_FILE
)

print(
    "Combined final file:",
    FINAL_FILTERED_FILE
)

Found CSV files: 355


/var/folders/64/7kp6gktj47s5xxbfxb3vs54m0000gn/T/ipykernel_13835/1698570049.py:45: DtypeWarning: Columns (71,75,77,79,81,85,115,127) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)



DATE FILTER FINISHED
Required period: 2020-01-01 to 2026-09-19
Stations processed: 355
Full coverage: 0
Missing start: 5
Missing end: 354
No valid date: 1

STATIONS THAT DO NOT COVER 2020-09-19
                     station_id                status original_start original_end  filtered_rows
                    USW00003822           MISSING_END     1948-01-01   2026-09-01             81
                    USW00004803           MISSING_END     1998-06-01   2026-09-01             81
                    USW00012907           MISSING_END     1965-03-01   2026-09-01             81
                    USW00003177           MISSING_END     1998-03-01   2026-08-01             80
                    USW00094790           MISSING_END     1949-05-01   2026-09-01             81
                    USW00093996           MISSING_END     1949-10-01   2026-08-01             19
                    USW00024090           MISSING_END     1948-05-01   2026-09-01             81
                    USW000938

In [25]:
from pathlib import Path
import pandas as pd

# ============================================================
# PATHS
# ============================================================

BASE_DIR = Path("/Users/zhouyuxuan/Downloads/airport_weather")

MATCH_FILE = BASE_DIR / "airport_station_matches.csv"

FILTERED_DIR = BASE_DIR / "filtered_2020_to_now"

OUTPUT_FILE = (
    BASE_DIR /
    "airport_GSOM_2020_to_now_FINAL.csv"
)


# ============================================================
# 1. READ AIRPORT -> STATION MATCH
# ============================================================

matches = pd.read_csv(MATCH_FILE)

print("Matched airports:", len(matches))

print(
    matches[
        [
            "airport_name",
            "station_id",
            "station_name",
            "station_type"
        ]
    ].head()
)


# ============================================================
# 2. ONLY KEEP AIRPORTS THAT HAVE DATA
# ============================================================

matches_valid = matches[
    matches["station_id"].notna()
].copy()

print(
    "Airports with matched stations:",
    len(matches_valid)
)


# ============================================================
# 3. READ ALL FILTERED GSOM FILES
# ============================================================

all_data = []

files = list(
    FILTERED_DIR.glob("*.csv")
)

print(
    "Filtered station files:",
    len(files)
)

for file in files:

    try:

        df = pd.read_csv(file)

        # Station ID from filename
        station_id = file.name.split("_2020_to_")[0]

        df["station_id"] = station_id

        all_data.append(df)

    except Exception as e:

        print(
            f"ERROR: {file.name} -> {e}"
        )


# ============================================================
# 4. COMBINE ALL STATIONS
# ============================================================

weather = pd.concat(
    all_data,
    ignore_index=True
)

print(
    "Weather rows:",
    len(weather)
)

print(
    "Stations:",
    weather["station_id"].nunique()
)


# ============================================================
# 5. MERGE WITH AIRPORT INFORMATION
#
# This adds the ORIGINAL airport_name
# from june_airports.xlsx
# ============================================================

airport_info = matches_valid[
    [
        "airport_name",
        "city",
        "state",
        "airport",
        "station_id",
        "station_name",
        "station_type",
        "latitude",
        "longitude",
        "elevation",
        "match_score"
    ]
].copy()


final = weather.merge(
    airport_info,
    on="station_id",
    how="inner"
)


# ============================================================
# 6. MAKE airport_name THE FIRST COLUMN
# ============================================================

first_columns = [
    "airport_name",
    "city",
    "state",
    "airport",
    "station_id",
    "station_name",
    "station_type",
    "latitude",
    "longitude",
    "elevation",
    "match_score",
    "DATE"
]

other_columns = [
    c for c in final.columns
    if c not in first_columns
]

final = final[
    first_columns + other_columns
]


# ============================================================
# 7. SORT
#
# airport_name -> DATE
# ============================================================

final["DATE"] = pd.to_datetime(
    final["DATE"],
    errors="coerce"
)

final = final.sort_values(
    ["airport_name", "DATE"]
).reset_index(drop=True)


# ============================================================
# 8. SAVE
# ============================================================

final.to_csv(
    OUTPUT_FILE,
    index=False
)


# ============================================================
# 9. CHECK
# ============================================================

print("\n====================================")
print("FINAL DATA CREATED")
print("====================================")

print(
    "File:",
    OUTPUT_FILE
)

print(
    "Rows:",
    len(final)
)

print(
    "Airports:",
    final["airport_name"].nunique()
)

print(
    "Stations:",
    final["station_id"].nunique()
)

print(
    "Date:",
    final["DATE"].min(),
    "to",
    final["DATE"].max()
)

print("\nFirst columns:")
print(final.columns[:15].tolist())

print("\nFirst 10 rows:")
print(
    final[
        [
            "airport_name",
            "station_id",
            "station_name",
            "DATE",
            "TAVG",
            "TMAX",
            "TMIN",
            "PRCP"
        ]
    ].head(10)
)

Matched airports: 368
                       airport_name   station_id           station_name  \
0   Aberdeen, SD: Aberdeen Regional  USC00220021               ABERDEEN   
1     Abilene, TX: Abilene Regional  USC00140010                ABILENE   
2             Adak Island, AK: Adak  USW00025704                ADAK AP   
3   Aguadilla, PR: Rafael Hernandez  USW00013837  AUGUSTA DANIEL FLD AP   
4  Akron, OH: Akron-Canton Regional  USW00014895        AKRON CANTON AP   

    station_type  
0  CITY_OR_OTHER  
1  CITY_OR_OTHER  
2        AIRPORT  
3        AIRPORT  
4        AIRPORT  
Airports with matched stations: 368
Filtered station files: 275


/var/folders/64/7kp6gktj47s5xxbfxb3vs54m0000gn/T/ipykernel_13835/2292125462.py:73: DtypeWarning: Columns (71,75,77,79,81,85,115,127) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


Weather rows: 41902
Stations: 275

FINAL DATA CREATED
File: /Users/zhouyuxuan/Downloads/airport_weather/airport_GSOM_2020_to_now_FINAL.csv
Rows: 22021
Airports: 288
Stations: 274
Date: 2020-01-01 00:00:00 to 2026-09-01 00:00:00

First columns:
['airport_name', 'city', 'state', 'airport', 'station_id', 'station_name', 'station_type', 'latitude', 'longitude', 'elevation', 'match_score', 'DATE', 'STATION', 'LATITUDE', 'LONGITUDE']

First 10 rows:
                      airport_name   station_id station_name       DATE  \
0  Aberdeen, SD: Aberdeen Regional  USC00220021     ABERDEEN 2020-01-01   
1  Aberdeen, SD: Aberdeen Regional  USC00220021     ABERDEEN 2020-02-01   
2  Aberdeen, SD: Aberdeen Regional  USC00220021     ABERDEEN 2020-03-01   
3  Aberdeen, SD: Aberdeen Regional  USC00220021     ABERDEEN 2020-04-01   
4  Aberdeen, SD: Aberdeen Regional  USC00220021     ABERDEEN 2020-05-01   
5  Aberdeen, SD: Aberdeen Regional  USC00220021     ABERDEEN 2020-06-01   
6  Aberdeen, SD: Aberdeen R

In [26]:
import pandas as pd
import numpy as np
import requests
import re
from pathlib import Path
from difflib import SequenceMatcher

# ============================================================
# PATHS
# ============================================================

AIRPORT_FILE = Path(
    "/Users/zhouyuxuan/Downloads/june_airports.xlsx"
)

OUTPUT_DIR = Path(
    "/Users/zhouyuxuan/Downloads/airport_weather"
)

MATCH_FILE = OUTPUT_DIR / "airport_station_matches_v2.csv"
TOP3_FILE = OUTPUT_DIR / "airport_station_top3_candidates.csv"

GSOM_URL = (
    "https://www.ncei.noaa.gov/data/"
    "global-summary-of-the-month/access/"
)

STATIONS_URL = (
    "https://www.ncei.noaa.gov/pub/data/ghcn/daily/"
    "ghcnd-stations.txt"
)


# ============================================================
# 1. READ YOUR 368 AIRPORTS
# ============================================================

airports = pd.read_excel(AIRPORT_FILE)

print("Airports:", len(airports))


# ============================================================
# 2. PARSE airport_name
# ============================================================

def parse_airport_name(x):

    x = str(x).strip()

    m = re.match(
        r"^(.*?),\s*([A-Z]{2}):\s*(.*)$",
        x
    )

    if m:
        return pd.Series({
            "city": m.group(1).strip(),
            "state": m.group(2).strip(),
            "airport": m.group(3).strip()
        })

    return pd.Series({
        "city": x,
        "state": "",
        "airport": ""
    })


airports[
    ["city", "state", "airport"]
] = airports["airport_name"].apply(
    parse_airport_name
)


# ============================================================
# 3. GET ALL REAL GSOM STATION IDS
# ============================================================

print("\nGetting GSOM station list...")

r = requests.get(
    GSOM_URL,
    timeout=60
)

r.raise_for_status()

gsom_files = re.findall(
    r'href="([A-Z0-9]+)\.csv"',
    r.text,
    flags=re.IGNORECASE
)

gsom_set = set(gsom_files)

print(
    "GSOM stations:",
    len(gsom_set)
)


# ============================================================
# 4. GET GHCN STATION METADATA
# ============================================================

print("\nDownloading GHCN station metadata...")

r = requests.get(
    STATIONS_URL,
    timeout=120
)

r.raise_for_status()

lines = r.text.splitlines()

stations = []

for line in lines:

    if len(line) < 71:
        continue

    station_id = line[0:11].strip()

    # --------------------------------------------------------
    # VERY IMPORTANT:
    # Only stations that ACTUALLY have GSOM files
    # --------------------------------------------------------

    if station_id not in gsom_set:
        continue

    try:

        latitude = float(
            line[12:20].strip()
        )

        longitude = float(
            line[21:30].strip()
        )

        elevation = float(
            line[31:37].strip()
        )

    except:

        latitude = np.nan
        longitude = np.nan
        elevation = np.nan

    # NOAA GHCN station format
    state = line[38:40].strip()

    name = line[41:71].strip()

    stations.append({
        "station_id": station_id,
        "latitude": latitude,
        "longitude": longitude,
        "elevation": elevation,
        "station_state": state,
        "station_name": name
    })


stations = pd.DataFrame(stations)

print(
    "GSOM stations with metadata:",
    len(stations)
)


# ============================================================
# 5. NORMALIZE TEXT
# ============================================================

def normalize(text):

    text = str(text).upper()

    replacements = {
        "INTERNATIONAL": "INTL",
        "AIRPORT": "AP",
        "REGIONAL": "REG",
        "MUNICIPAL": "MUNI",
        "COUNTY": "",
        "FIELD": "FLD",
        "SAWYER": "",
        "STATION": "",
    }

    for old, new in replacements.items():

        text = text.replace(
            old,
            new
        )

    text = re.sub(
        r"[^A-Z0-9 ]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


# ============================================================
# 6. IDENTIFY BAD DISTANCE STATIONS
#
# Examples:
# BELLEVILLE 2.2 SE
# CHICAGO 6.6 E
# etc.
# ============================================================

def is_distance_station(name):

    name = str(name).upper()

    pattern = (
        r"\d+(?:\.\d+)?\s*"
        r"(?:N|S|E|W)"
        r"(?:\s|$)"
    )

    return bool(
        re.search(
            pattern,
            name
        )
    )


stations["distance_station"] = (
    stations["station_name"]
    .apply(is_distance_station)
)


# ============================================================
# 7. IDENTIFY AIRPORT STATIONS
# ============================================================

AIRPORT_KEYWORDS = [
    "AIRPORT",
    " AP",
    "INTL",
    "INTERNATIONAL",
    "REGIONAL",
    "FIELD",
    "MUNICIPAL",
    "NAS",
    "AFB",
    "AIR FORCE",
]


def is_airport_station(name):

    name = str(name).upper()

    for keyword in AIRPORT_KEYWORDS:

        if keyword in name:
            return True

    return False


stations["is_airport_station"] = (
    stations["station_name"]
    .apply(is_airport_station)
)


# ============================================================
# 8. REMOVE BAD STATION TYPES
# ============================================================

candidates_base = stations.copy()

# Remove distance stations
candidates_base = candidates_base[
    ~candidates_base["distance_station"]
]

# Remove citizen weather stations
# US1 = personal/citizen network
candidates_base = candidates_base[
    ~candidates_base["station_id"].str.startswith("US1")
]

print(
    "Usable candidate stations:",
    len(candidates_base)
)


# ============================================================
# 9. SIMILARITY FUNCTION
# ============================================================

def similarity(a, b):

    a = normalize(a)
    b = normalize(b)

    if not a or not b:
        return 0

    return SequenceMatcher(
        None,
        a,
        b
    ).ratio()


# ============================================================
# 10. MATCH ONE AIRPORT
# ============================================================

def get_candidates(row):

    city = str(row["city"])
    state = str(row["state"])
    airport = str(row["airport"])

    # --------------------------------------------------------
    # FIRST: SAME STATE ONLY
    # --------------------------------------------------------

    candidates = candidates_base[
        candidates_base["station_state"] == state
    ].copy()

    # If no same-state candidate, return empty
    if len(candidates) == 0:
        return pd.DataFrame()

    # --------------------------------------------------------
    # Normalize airport information
    # --------------------------------------------------------

    city_norm = normalize(city)
    airport_norm = normalize(airport)

    airport_words = [
        word
        for word in airport_norm.split()
        if len(word) >= 4
    ]

    scores = []

    for _, station in candidates.iterrows():

        station_name = station["station_name"]

        station_norm = normalize(
            station_name
        )

        # ----------------------------------------------------
        # City similarity
        # ----------------------------------------------------

        city_score = similarity(
            city,
            station_name
        )

        # ----------------------------------------------------
        # Airport-name similarity
        # ----------------------------------------------------

        airport_score = similarity(
            airport,
            station_name
        )

        # ----------------------------------------------------
        # Base score
        # ----------------------------------------------------

        score = (
            0.45 * city_score +
            0.35 * airport_score
        )

        # ----------------------------------------------------
        # EXACT CITY MATCH
        # ----------------------------------------------------

        if city_norm in station_norm:

            score += 0.15

        # ----------------------------------------------------
        # AIRPORT STATION BONUS
        # ----------------------------------------------------

        if station["is_airport_station"]:

            score += 0.25

        # ----------------------------------------------------
        # AIRPORT NAME WORD MATCH
        # ----------------------------------------------------

        matched_words = 0

        for word in airport_words:

            if word in station_norm:
                matched_words += 1

        score += (
            0.05 * matched_words
        )

        # ----------------------------------------------------
        # USW airport network bonus
        # ----------------------------------------------------

        if station["station_id"].startswith("USW"):

            score += 0.10

        scores.append({
            "station_id": station["station_id"],
            "station_name": station["station_name"],
            "station_state": station["station_state"],
            "latitude": station["latitude"],
            "longitude": station["longitude"],
            "elevation": station["elevation"],
            "is_airport_station": station[
                "is_airport_station"
            ],
            "city_score": city_score,
            "airport_score": airport_score,
            "match_score": score
        })

    result = pd.DataFrame(scores)

    result = result.sort_values(
        "match_score",
        ascending=False
    ).reset_index(drop=True)

    return result


# ============================================================
# 11. GET TOP 3 FOR EVERY AIRPORT
# ============================================================

print("\nMatching airports...")

all_top3 = []
best_matches = []

for _, row in airports.iterrows():

    result = get_candidates(row)

    if len(result) == 0:

        best_matches.append({
            "airport_name": row["airport_name"],
            "city": row["city"],
            "state": row["state"],
            "airport": row["airport"],
            "station_id": None,
            "station_name": None,
            "station_state": None,
            "latitude": np.nan,
            "longitude": np.nan,
            "elevation": np.nan,
            "is_airport_station": False,
            "city_score": np.nan,
            "airport_score": np.nan,
            "match_score": np.nan
        })

        continue

    # --------------------------------------------------------
    # TOP 3
    # --------------------------------------------------------

    top3 = result.head(3).copy()

    top3.insert(
        0,
        "airport_name",
        row["airport_name"]
    )

    top3.insert(
        1,
        "airport_city",
        row["city"]
    )

    top3.insert(
        2,
        "airport_state",
        row["state"]
    )

    top3.insert(
        3,
        "airport",
        row["airport"]
    )

    top3["candidate_rank"] = (
        range(1, len(top3) + 1)
    )

    all_top3.append(top3)

    # --------------------------------------------------------
    # BEST MATCH
    # --------------------------------------------------------

    best = result.iloc[0]

    best_matches.append({
        "airport_name": row["airport_name"],
        "city": row["city"],
        "state": row["state"],
        "airport": row["airport"],
        "station_id": best["station_id"],
        "station_name": best["station_name"],
        "station_state": best["station_state"],
        "latitude": best["latitude"],
        "longitude": best["longitude"],
        "elevation": best["elevation"],
        "is_airport_station": best[
            "is_airport_station"
        ],
        "city_score": best["city_score"],
        "airport_score": best["airport_score"],
        "match_score": best["match_score"]
    })


# ============================================================
# 12. SAVE TOP 3
# ============================================================

top3_df = pd.concat(
    all_top3,
    ignore_index=True
)

top3_df.to_csv(
    TOP3_FILE,
    index=False
)


# ============================================================
# 13. SAVE BEST MATCH
# ============================================================

matches_v2 = pd.DataFrame(
    best_matches
)

matches_v2.to_csv(
    MATCH_FILE,
    index=False
)


# ============================================================
# 14. CHECK STATE MATCHING
# ============================================================

matches_v2["state_correct"] = (
    matches_v2["state"]
    ==
    matches_v2["station_state"]
)


# ============================================================
# 15. SUMMARY
# ============================================================

print("\n========================================")
print("MATCHING V2 FINISHED")
print("========================================")

print(
    "Airports:",
    len(matches_v2)
)

print(
    "Matched:",
    matches_v2["station_id"].notna().sum()
)

print(
    "State correct:",
    matches_v2["state_correct"].sum()
)

print(
    "State WRONG:",
    (
        ~matches_v2["state_correct"]
    ).sum()
)

print(
    "Airport stations:",
    matches_v2[
        "is_airport_station"
    ].sum()
)


# ============================================================
# 16. SHOW FIRST 30 MATCHES
# ============================================================

print("\n========================================")
print("FIRST 30 MATCHES")
print("========================================")

print(
    matches_v2[
        [
            "airport_name",
            "state",
            "station_id",
            "station_name",
            "station_state",
            "is_airport_station",
            "match_score"
        ]
    ].head(30).to_string(index=False)
)


# ============================================================
# 17. SHOW LOW-CONFIDENCE MATCHES
# ============================================================

print("\n========================================")
print("LOW CONFIDENCE MATCHES")
print("========================================")

low_conf = matches_v2[
    matches_v2["match_score"] < 0.70
]

print(
    low_conf[
        [
            "airport_name",
            "state",
            "station_id",
            "station_name",
            "is_airport_station",
            "match_score"
        ]
    ].to_string(index=False)
)


# ============================================================
# 18. SHOW EXAMPLES OF TOP 3
# ============================================================

print("\n========================================")
print("TOP 3 CANDIDATES EXAMPLES")
print("========================================")

print(
    top3_df[
        [
            "airport_name",
            "airport_state",
            "candidate_rank",
            "station_id",
            "station_name",
            "station_state",
            "is_airport_station",
            "match_score"
        ]
    ].head(30).to_string(index=False)
)


print("\n========================================")
print("FILES SAVED")
print("========================================")

print(
    "Best matches:",
    MATCH_FILE
)

print(
    "Top 3 candidates:",
    TOP3_FILE
)

Airports: 368

Getting GSOM station list...
GSOM stations: 126536

GSOM stations with metadata: 126536
Usable candidate stations: 76635

Matching airports...

MATCHING V2 FINISHED
Airports: 368
Matched: 365
State correct: 365
State WRONG: 3
Airport stations: 313

FIRST 30 MATCHES
                                                       airport_name state  station_id                   station_name station_state  is_airport_station  match_score
                                    Aberdeen, SD: Aberdeen Regional    SD USW00014929                       ABERDEEN            SD               False     1.030000
                                      Abilene, TX: Abilene Regional    TX USW00013962                ABILENE RGNL AP            TX                True     1.105594
                                              Adak Island, AK: Adak    AK USW00025704                        ADAK AP            AK                True     0.954545
                                    Aguadilla, PR: Rafael Herna

In [29]:
import pandas as pd
from pathlib import Path

# ============================================================
# 文件路径
# ============================================================

MATCH_FILE = Path(
    "/Users/zhouyuxuan/Downloads/airport_weather/airport_station_matches_v2.csv"
)

WEATHER_DIR = Path(
    "/Users/zhouyuxuan/Downloads/airport_weather/filtered_2020_to_now"
)

OUTPUT_FILE = Path(
    "/Users/zhouyuxuan/Downloads/airport_weather/airport_GSOM_2020_to_now_FINAL.csv"
)


# ============================================================
# 1. 读取你现在这个 MATCH 表
# ============================================================

matches = pd.read_csv(MATCH_FILE)

print("Match rows:", len(matches))

print("\nMatch columns:")
print(matches.columns.tolist())


# ============================================================
# 2. 只保留并表需要的信息
# ============================================================

match_info = matches[
    [
        "airport_name",
        "city",
        "state",
        "airport",
        "station_id",
        "station_name",
        "station_state",
        "latitude",
        "longitude",
        "elevation",
        "is_airport_station",
        "city_score",
        "airport_score",
        "match_score"
    ]
].copy()


# station_id 统一格式
match_info["station_id"] = (
    match_info["station_id"]
    .astype(str)
    .str.strip()
)


# ============================================================
# 3. 读取已经下载好的 weather CSV
# ============================================================

weather_files = list(
    WEATHER_DIR.glob("*.csv")
)

print(
    "\nWeather files found:",
    len(weather_files)
)


weather_list = []

for file in weather_files:

    station_id = file.stem

    df = pd.read_csv(file)

    # 给天气数据加 station_id
    df["station_id"] = station_id

    weather_list.append(df)


# 合并所有 station weather
weather = pd.concat(
    weather_list,
    ignore_index=True
)

print(
    "Total weather rows:",
    len(weather)
)


# ============================================================
# 4. 用 station_id 并表
# ============================================================

final = match_info.merge(
    weather,
    on="station_id",
    how="inner"
)


# ============================================================
# 5. airport_name 放第一列
# ============================================================

first_columns = [
    "airport_name",
    "city",
    "state",
    "airport",
    "station_id",
    "station_name",
    "station_state",
    "latitude",
    "longitude",
    "elevation",
    "is_airport_station",
    "city_score",
    "airport_score",
    "match_score",
    "DATE"
]

first_columns = [
    x for x in first_columns
    if x in final.columns
]

other_columns = [
    x for x in final.columns
    if x not in first_columns
]

final = final[
    first_columns + other_columns
]


# ============================================================
# 6. 排序
# ============================================================

final = final.sort_values(
    ["airport_name", "DATE"]
).reset_index(drop=True)


# ============================================================
# 7. 保存
# ============================================================

final.to_csv(
    OUTPUT_FILE,
    index=False
)


# ============================================================
# 8. 检查结果
# ============================================================

print("\n====================================")
print("DONE")
print("====================================")

print(
    "Output:",
    OUTPUT_FILE
)

print(
    "Rows:",
    len(final)
)

print(
    "Unique airports:",
    final["airport_name"].nunique()
)

print(
    "Unique stations:",
    final["station_id"].nunique()
)

print("\nFirst columns:")
print(
    final.columns[:20].tolist()
)

print("\nPreview:")
print(
    final.head()
)

Match rows: 368

Match columns:
['airport_name', 'city', 'state', 'airport', 'station_id', 'station_name', 'station_state', 'latitude', 'longitude', 'elevation', 'is_airport_station', 'city_score', 'airport_score', 'match_score']

Weather files found: 275


/var/folders/64/7kp6gktj47s5xxbfxb3vs54m0000gn/T/ipykernel_13835/3763954745.py:85: DtypeWarning: Columns (71,75,77,79,81,85,115,127) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


Total weather rows: 41902

DONE
Output: /Users/zhouyuxuan/Downloads/airport_weather/airport_GSOM_2020_to_now_FINAL.csv
Rows: 0
Unique airports: 0
Unique stations: 0

First columns:
['airport_name', 'city', 'state', 'airport', 'station_id', 'station_name', 'station_state', 'latitude', 'longitude', 'elevation', 'is_airport_station', 'city_score', 'airport_score', 'match_score', 'DATE', 'STATION', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME']

Preview:
Empty DataFrame
Columns: [airport_name, city, state, airport, station_id, station_name, station_state, latitude, longitude, elevation, is_airport_station, city_score, airport_score, match_score, DATE, STATION, LATITUDE, LONGITUDE, ELEVATION, NAME, ADPT, ADPT_ATTRIBUTES, ASLP, ASLP_ATTRIBUTES, ASTP, ASTP_ATTRIBUTES, AWBT, AWBT_ATTRIBUTES, AWND, AWND_ATTRIBUTES, CDSD, CDSD_ATTRIBUTES, CLDD, CLDD_ATTRIBUTES, DP01, DP01_ATTRIBUTES, DP10, DP10_ATTRIBUTES, DP1X, DP1X_ATTRIBUTES, DSND, DSND_ATTRIBUTES, DSNW, DSNW_ATTRIBUTES, DT00, DT00_ATTRIBUTES,

In [30]:
import pandas as pd

# 读取数据
df1 = pd.read_csv("/Users/zhouyuxuan/Downloads/airport_weather/airport_station_top3_candidates.csv")

# 要检查的列名
column_name = "candidate_rank"

# 统计值为 1 的行数
count_1 = (df1[column_name] == 1).sum()

# 输出结果
print(f"列 {column_name} 中值为 1 的行数: {count_1}")

列 candidate_rank 中值为 1 的行数: 365


In [34]:
import pandas as pd

# 读取两个表
table1 = pd.read_csv("/Users/zhouyuxuan/Downloads/airport_weather/airport_station_top3_candidates.csv")
table2 = pd.read_csv("/Users/zhouyuxuan/Downloads/airport_weather/airport_GSOM_2020_to_2026-09-19.csv")

# 只保留表格1中 rank = 1 的记录
rank1 = table1[table1["candidate_rank"] == 1][["station_id", "airport_name"]]

# 按 station_id 对应
table2 = table2.merge(
    rank1,
    on="station_id",
    how="left"
)

# 把 airport_name 移到第一列
cols = ["airport_name"] + [col for col in table2.columns if col != "airport_name"]
table2 = table2[cols]

# 保存结果
output_path = "/Users/zhouyuxuan/Downloads/table2_with_airport.csv"
table2.to_csv(output_path, index=False)

print("保存成功：", output_path)
print(table2.head())

/var/folders/64/7kp6gktj47s5xxbfxb3vs54m0000gn/T/ipykernel_13835/2306220299.py:5: DtypeWarning: Columns (71,75,77,79,81,85,115,127) have mixed types. Specify dtype option on import or set low_memory=False.
  table2 = pd.read_csv("/Users/zhouyuxuan/Downloads/airport_weather/airport_GSOM_2020_to_2026-09-19.csv")


保存成功： /Users/zhouyuxuan/Downloads/table2_with_airport.csv
                                       airport_name      STATION        DATE  \
0  Savannah, GA: Savannah/Hilton Head International  USW00003822  2020-01-01   
1  Savannah, GA: Savannah/Hilton Head International  USW00003822  2020-02-01   
2  Savannah, GA: Savannah/Hilton Head International  USW00003822  2020-03-01   
3  Savannah, GA: Savannah/Hilton Head International  USW00003822  2020-04-01   
4  Savannah, GA: Savannah/Hilton Head International  USW00003822  2020-05-01   

   LATITUDE  LONGITUDE  ELEVATION  \
0  32.13133   -81.2023        8.7   
1  32.13133   -81.2023        8.7   
2  32.13133   -81.2023        8.7   
3  32.13133   -81.2023        8.7   
4  32.13133   -81.2023        8.7   

                                                NAME   ADPT ADPT_ATTRIBUTES  \
0  SAVANNAH HILTON HEAD INTERNATIONAL AIRPORT, GA US   7.60            ,,,W   
1  SAVANNAH HILTON HEAD INTERNATIONAL AIRPORT, GA US   8.34            ,,,W   
2

In [35]:
# 统计不同的 airport_name 数量
count = table2["airport_name"].nunique()

print("不同的 airport_name 数量：", count)

不同的 airport_name 数量： 235
